# 02 · Avaliação da fase 2 nos 8 cenários do AIT-ADS

Pergunta: o pipeline da fase 2 (correlação, motor de governança e política anti-fadiga) reduz
os alertas a poucos incidentes e a poucas mensagens **sem perder o ataque**?

- **O que roda:** o adaptador do Wazuh, a correlação, o motor e o roteador de notificação, pelas
  mesmas funções puras do worker (`ronda.avaliacao.Simulador`), em memória. Sem banco: são
  2,3 milhões de eventos, e o objetivo é medir as regras, não o PostgreSQL.
- **Relógio:** o do evento. Quatro dias de cenário passam por quatro noites de horário de silêncio
  e quatro resumos diários.
- **Canal:** falso (`CanalFalso`). Nada vai para o Telegram; as mensagens são montadas pelos
  templates de verdade e guardadas.
- **Política:** `policies/avaliacao/ait-ads.yaml`, que é a de demonstração **sem ajuste**, com o
  inventário de cada cenário tirado do CSV (host → IP) e a criticidade por papel de
  `ronda.avaliacao.ait_ads.CRITICIDADE_POR_PAPEL`.
- **Rótulos:** `time_label` (a janela da fase) e `event_label` (o ataque causou o alerta), do CSV
  dos autores, casados pela posição e conferidos pelo horário.

Como rodar: `uv sync --extra notebooks` e `uv run jupyter lab`, a partir da raiz. Os dados não
estão no git; veja `data/README.md`. O notebook é versionado **sem saídas**; os números estão em
`docs/avaliacao-fase2.md`.

In [ ]:
import time
from pathlib import Path

import pandas as pd

from ronda.avaliacao import CENARIOS, avaliar_cenario

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 60)
pd.set_option("display.max_rows", 200)

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
RAW = RAIZ / "data" / "raw" / "ait-ads"
POLITICA = RAIZ / "policies" / "avaliacao" / "ait-ads.yaml"
assert (RAW / "json" / "fox_wazuh.json").exists(), "dados ausentes: veja data/README.md"

## 1. Rodar os 8 cenários

Cerca de 10 minutos no total; o fox, o wheeler, o harrison e o wilson passam de 400 mil eventos
cada, por causa do `dirb`.

In [ ]:
resultados = {}
for cenario in CENARIOS:
    inicio = time.perf_counter()
    resultados[cenario] = avaliar_cenario(RAW, cenario, POLITICA)
    print(f"{cenario:16} {time.perf_counter() - inicio:6.0f} s")

## 2. Redução: eventos → incidentes → mensagens

- **eventos:** alertas convertidos (sem as cópias do Suricata vindas do `fast.log`);
- **notificações:** subidas de prioridade que a política manda avisar (P1, P2 e P3);
- **só no painel:** subidas para P4, que nunca notificam;
- **mensagens:** o que chegaria ao Telegram, depois do agrupamento do P2 e do resumo do P3.

In [ ]:
resumo = pd.DataFrame([r.resumo() for r in resultados.values()]).set_index("cenario")
total = resumo.sum(numeric_only=True)
total["eventos_por_incidente"] = round(total["eventos"] / total["incidentes"], 1)
total["eventos_por_mensagem"] = round(total["eventos"] / total["mensagens"])
resumo.loc["total"] = total
colunas = [
    "alertas",
    "eventos",
    "incidentes",
    "decisoes",
    "notificacoes",
    "suprimidas",
    "so_painel",
    "mensagens",
    "mensagens_P1_P2",
    "mensagens_resumo",
    "eventos_por_incidente",
    "eventos_por_mensagem",
]
resumo[colunas].astype({c: "int64" for c in colunas if c != "eventos_por_incidente"})

In [ ]:
# De onde veio o evento que levou cada incidente a P1/P2: causado pelo ataque, só na janela de
# uma fase, ou fora de qualquer ataque (ruído que acorda alguém).
resumo[
    [
        "incidentes_P1",
        "incidentes_P2",
        "incidentes_P3",
        "incidentes_P4",
        "graves_pela_causa",
        "graves_na_janela",
        "graves_fora_de_ataque",
    ]
].astype("int64")

## 3. Mensagens e espera

A espera vai do registro da notificação ao envio: o P1 sai no despacho seguinte, mesmo de
madrugada; o P2 espera a janela do agrupamento (10 min) e o fim do horário de silêncio; o P3
espera o resumo das 08:00.

In [ ]:
pd.DataFrame({c: r.mensagens_detalhadas() for c, r in resultados.items()}).fillna(0).astype(int)

In [ ]:
espera = [
    {"cenario": c, "prioridade": p, **v}
    for c, r in resultados.items()
    for p, v in r.espera().items()
]
pd.DataFrame(espera).set_index(["prioridade", "cenario"]).sort_index()

## 4. Cada fase do ataque gerou um incidente P1 ou P2?

Três leituras, da mais frouxa à mais estrita, porque "o incidente contém um evento da fase"
engana: um incidente que já era P1 antes do ataque absorve os eventos da fase sem que ninguém
receba aviso novo.

- **pela janela:** a prioridade mais grave entre os incidentes com algum evento da janela da fase;
- **pela causa:** a mesma coisa, só com os eventos que o ataque causou (`event_label`);
- **gerou:** o evento que levou o incidente a P1/P2 é da fase (`causa` quando o ataque o causou,
  `janela` quando só caiu na janela). É a leitura que responde à pergunta.

O diagnóstico separa o que é da política do que é da fonte: *sem alerta causado pelo ataque* quer
dizer que nem o Wazuh nem o Suricata viram a fase, e nenhuma política conserta isso.

In [ ]:
cobertura = pd.DataFrame([linha for r in resultados.values() for linha in r.cobertura()])
cobertura[
    [
        "cenario",
        "fase",
        "eventos",
        "causados",
        "pela_janela",
        "pela_causa",
        "gerou",
        "diagnostico",
        "incidente_gerado",
    ]
]

In [ ]:
# ✅ gerou P1/P2 · ↪ caiu num P1/P2 que já existia · ◌ P1/P2 só na janela ·
# P3/P4 vista com essa prioridade · ✗ nenhum alerta causado pelo ataque
MARCAS = {
    "gerou P1/P2": "✅",
    "caiu num P1/P2 que já existia": "↪",
    "P1/P2 na janela, sem causa": "◌",
    "sem alerta causado pelo ataque": "✗",
}


def marca(diagnostico):
    if not isinstance(diagnostico, str):
        return ""
    return MARCAS.get(diagnostico, diagnostico.removeprefix("vista, mas "))


tabela = cobertura.pivot(index="fase", columns="cenario", values="diagnostico").map(marca)
ordem = [
    "network_scans",
    "service_scans",
    "wpscan",
    "dirb",
    "webshell",
    "cracking",
    "reverse_shell",
    "privilege_escalation",
    "service_stop",
    "dnsteal",
]
tabela.reindex([f for f in ordem if f in tabela.index])

In [ ]:
cobertura.groupby("fase")["diagnostico"].value_counts().unstack(fill_value=0)

## 5. Os P1 e P2 que acordariam alguém

Um incidente P1/P2 **sem nenhum evento dentro de uma janela de ataque** é ruído que chega com
urgência: é o número que a política anti-fadiga precisa manter baixo.

In [ ]:
graves = pd.DataFrame([g for r in resultados.values() for g in r.graves()])
graves.groupby(["cenario", "prioridade", "categoria", "ativo", "origem"]).agg(
    incidentes=("eventos", "size"),
    eventos=("eventos", "sum"),
    pela_causa=("subida", lambda s: int((s == "causa").sum())),
    fora_de_ataque=("subida", lambda s: int((s == "fora").sum())),
).sort_values("incidentes", ascending=False)

In [ ]:
graves.groupby(["categoria", "subida"]).size().unstack(fill_value=0)

## 6. Supressões

O silêncio por origem só age quando um incidente novo da mesma chave aparece dentro da janela de
silêncio contada da notificação anterior. Com `silencio_por_origem` igual à janela da correlação
(30 min nas duas), isso quase não acontece: para nascer um incidente novo, a chave precisa ficar
30 min sem evento, e a notificação anterior é, no mínimo, desse tempo atrás.

In [ ]:
supressoes = {c: dict(r.simulador.contagens.motivos_supressao) for c, r in resultados.items()}
pd.DataFrame(supressoes).fillna(0)

## 7. Uma mensagem de exemplo

A primeira mensagem P1 de cada cenário, como sairia no Telegram.

In [ ]:
for cenario, r in resultados.items():
    p1 = next((m for m in r.simulador.canal.enviadas if m.texto.startswith("🔴")), None)
    if p1:
        print(f"--- {cenario} ---\n{p1.texto}\n")
        break